# NB02: Tile coordinate extraction

Extracts tissue-tile coordinates from TCGA whole-slide images. Saves
coordinate CSVs only (no pixel data). NB03 reads pixels on demand from
each WSI using these coordinates.

**Manuscript parameters** (Methods, Supp Table S5/S6):
- Tile size: 256 x 256 px
- Stride: 512 px
- Magnification: 20x (level 0; ~0.5 um/px)
- Maximum tiles per slide: 2,000
- Tissue requirement: >= 30% per tile region
- Total tiles after extraction: 27,608,061 (across 19,996 slides)

**Required env**: `WSI_ROOT`, `WORKSPACE`.
**Outputs**: `coords/{slide_id}.csv` per slide; `runs/tile_<ts>/{summary.csv, gate_report.json}`.

In [ ]:
import os
import time
import json
import gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from contextlib import contextmanager

import numpy as np
import pandas as pd
import openslide
import cv2

# tame numpy/cv2 thread oversubscription under the thread pool
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
cv2.setNumThreads(1)

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
WSI_ROOT = Path(os.environ["WSI_ROOT"]).resolve()
COORDS_DIR = WORKSPACE / "coords"
RUNS_DIR = WORKSPACE / "runs"
COORDS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked parameters
PATCH_SIZE = 256
STRIDE = 512
MAX_TILES = 2000
MIN_TISSUE_FRAC = 0.30
LEVEL = 0
TISSUE_LEVEL = 3       # downsampled level for tissue masking only
NUM_WORKERS = 20
WSI_EXTS = (".svs", ".tif", ".tiff", ".ndpi", ".mrxs", ".scn", ".bif", ".svslide")

RUN_NAME = f"tile_{datetime.now():%Y%m%d_%H%M%S}_L{LEVEL}_ps{PATCH_SIZE}_st{STRIDE}_cap{MAX_TILES}"
RUN_DIR = RUNS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR / "run.log"

def log_line(msg):
    ts = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
    with open(LOG_PATH, "a") as f:
        f.write(ts + msg + "\n")
    print(msg)

log_line(f"WSI_ROOT   = {WSI_ROOT}")
log_line(f"COORDS_DIR = {COORDS_DIR}")
log_line(f"params     : PATCH_SIZE={PATCH_SIZE}, STRIDE={STRIDE}, LEVEL={LEVEL}, "
         f"MAX_TILES={MAX_TILES}, MIN_TISSUE_FRAC={MIN_TISSUE_FRAC}")


In [ ]:
# tissue mask via HSV; conservative threshold to exclude background and pen marks
def tissue_mask_rgb(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1] / 255.0
    val = hsv[:, :, 2] / 255.0
    return (sat > 0.10) & (val < 0.98)

def slide_id_from_path(p):
    return Path(p).stem

def find_all_wsis(root):
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in WSI_EXTS)


In [ ]:
def level_for_mask(slide, prefer_level=TISSUE_LEVEL):
    return min(max(0, prefer_level), slide.level_count - 1)

def read_thumbnail(slide, level):
    w, h = slide.level_dimensions[level]
    max_pix = 6000 * 6000
    if w * h > max_pix:
        scale = (max_pix / (w * h)) ** 0.5
        w, h = int(w * scale), int(h * scale)
    img = slide.get_thumbnail((w, h)).convert("RGB")
    return np.array(img)

def grid_coords(slide, mask_level, stride_l0, patch_size_l0, min_tissue_frac, max_tiles):
    """Return up to max_tiles (x, y) coordinates at level 0 for tiles meeting tissue threshold."""
    lvl_down = float(slide.level_downsamples[mask_level])
    win = int(round(patch_size_l0 / lvl_down))
    step = max(1, int(round(stride_l0 / lvl_down)))

    thumb = read_thumbnail(slide, mask_level)
    mask = tissue_mask_rgb(thumb)
    H, W = mask.shape
    lvl_w, lvl_h = slide.level_dimensions[mask_level]
    sx, sy = W / float(lvl_w), H / float(lvl_h)

    coords = []
    tissue_thresh = int(min_tissue_frac * (win * win))
    for yy in range(0, H - win + 1, step):
        row = mask[yy:yy + win, :]
        for xx in range(0, W - win + 1, step):
            sub = row[:, xx:xx + win]
            if int(np.count_nonzero(sub)) >= tissue_thresh:
                x0 = int(round((xx / sx) * lvl_down))
                y0 = int(round((yy / sy) * lvl_down))
                coords.append((x0, y0))
                if max_tiles and len(coords) >= max_tiles:
                    return coords
    return coords

def write_coords_csv(out_csv, coords, patch_size):
    if not coords:
        out_csv.write_text("x,y,w,h,level\n")
        return
    df = pd.DataFrame(coords, columns=["x", "y"])
    df["w"] = patch_size
    df["h"] = patch_size
    df["level"] = 0
    df.to_csv(out_csv, index=False)


In [ ]:
@contextmanager
def timing():
    t0 = time.time()
    yield lambda: time.time() - t0

def process_one(slide_path):
    sid = slide_id_from_path(slide_path)
    out_csv = COORDS_DIR / f"{sid}.csv"
    if out_csv.exists():
        return {"slide_id": sid, "n_patches": None, "seconds": 0.0, "skipped": True}

    slide = None
    try:
        with timing() as elapsed:
            slide = openslide.OpenSlide(str(slide_path))
            lvl = level_for_mask(slide, TISSUE_LEVEL)
            coords = grid_coords(slide, lvl, STRIDE, PATCH_SIZE, MIN_TISSUE_FRAC, MAX_TILES)
            n = len(coords)
            write_coords_csv(out_csv, coords, PATCH_SIZE)
            sec = elapsed()
        return {"slide_id": sid, "n_patches": n, "seconds": sec, "skipped": False, "path": str(slide_path)}
    except Exception as e:
        return {"slide_id": sid, "n_patches": -1, "seconds": 0.0, "error": repr(e), "path": str(slide_path)}
    finally:
        if slide is not None:
            try:
                slide.close()
            except Exception:
                pass
        gc.collect()


In [ ]:
# build worklist; skip slides that already have coord CSVs (resume-safe)
all_wsis = find_all_wsis(WSI_ROOT)
existing = {p.stem for p in COORDS_DIR.glob("*.csv")}
todo = [p for p in all_wsis if slide_id_from_path(p) not in existing]

log_line(f"WSIs found: {len(all_wsis):,} | already done: {len(existing):,} | to process: {len(todo):,}")

cfg = {
    "patch_size": PATCH_SIZE,
    "stride": STRIDE,
    "max_tiles": MAX_TILES,
    "min_tissue_frac": MIN_TISSUE_FRAC,
    "level": LEVEL,
    "tissue_level": TISSUE_LEVEL,
    "num_workers": NUM_WORKERS,
    "wsi_root": str(WSI_ROOT),
    "coords_dir": str(COORDS_DIR),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
}
(RUN_DIR / "config.json").write_text(json.dumps(cfg, indent=2))


In [ ]:
# parallel run
summary_csv = RUN_DIR / "summary.csv"
if not summary_csv.exists():
    pd.DataFrame(columns=["slide_id", "seconds", "n_patches", "path"]).to_csv(summary_csv, index=False)

n_done, n_err = 0, 0
total = len(todo)
t_start = time.time()

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futs = {ex.submit(process_one, p): p for p in todo}
    for i, fut in enumerate(as_completed(futs), 1):
        res = fut.result()
        sid = res["slide_id"]
        if "error" in res:
            n_err += 1
            log_line(f"ERROR {sid} {res['error']}")
        elif not res.get("skipped"):
            n_done += 1
            pd.DataFrame(
                [[sid, float(res["seconds"]), int(res["n_patches"]), res.get("path", "")]],
                columns=["slide_id", "seconds", "n_patches", "path"],
            ).to_csv(summary_csv, mode="a", header=False, index=False)
            log_line(f"OK {sid} patches={res['n_patches']} time_sec={res['seconds']:.1f}")
        if i % 50 == 0 or i == total:
            elapsed_h = (time.time() - t_start) / 3600
            log_line(f"...progress: {i}/{total} | elapsed {elapsed_h:.2f} h")

elapsed_h = (time.time() - t_start) / 3600
log_line(f"done: {n_done} processed, {n_err} errors, wall-time {elapsed_h:.2f} h")


In [ ]:
# acceptance gate
summary = pd.read_csv(summary_csv)
total_slides = len(all_wsis)
done_slides = len(summary)
total_patches = int(summary["n_patches"].fillna(0).sum())
mean_per_slide = float(summary["n_patches"].fillna(0).mean()) if done_slides else 0.0

gate = {
    "run": RUN_NAME,
    "slides_total": total_slides,
    "slides_processed": done_slides,
    "total_patches": total_patches,
    "mean_patches_per_slide": round(mean_per_slide, 1),
    "patch_cap": MAX_TILES,
    "params": cfg,
}
(RUN_DIR / "gate_report.json").write_text(json.dumps(gate, indent=2))

print()
print(f"slides processed : {done_slides:,} / {total_slides:,}")
print(f"total tiles      : {total_patches:,}")
print(f"mean per slide   : {mean_per_slide:.1f}")
print(f"manuscript ref   : 27,608,061 tiles across 19,996 slides (Supp S6)")
print(f"coords saved to  : {COORDS_DIR}")
